#Fine-Tuning GPT-3

Copyright 2024 Denis Rothman

[OpenAI fine-tuning documentation](https://beta.openai.com/docs/guides/fine-tuning/)

Check the cost of fine-tuning your dataset on OpenAI before running the notebook.

Run this notebook cell by cell to:

1.prepare data
2.fine-tune a model
3.run a fine-tuned model
4.manage the fine-tunes

## Installing OpenAI & Wandb

Restart the runtime after installing openai and run the cell again to make sur that "import openai" is executed.

In [47]:
try:
  import openai
except:
  !pip install openai
  import openai

## Your API Key

In [48]:
#You can retrieve your API key from a file(1)
# or enter it manually(2)

#Comment this cell if you want to enter your key manually.
#(1)Retrieve the API Key from a file
#Store you key in a file and read it(you can type it directly in the notebook but it will be visible for somebody next to you)
# from google.colab import drive
# drive.mount('/content/drive')
# f = open("drive/MyDrive/files/api_key.txt", "r")
# API_KEY=f.readline()
# f.close()

In [49]:
# #remove when repository goes public
# with open('drive/MyDrive/files/github.txt', 'r') as f:
#     github_token = f.read().strip()

In [50]:
#(2) Enter your manually by
# replacing API_KEY by your key.
#The OpenAI Key
import os
os.environ['OPENAI_API_KEY']
openai.api_key = os.getenv("OPENAI_API_KEY")

In [51]:
try:
  import wandb
except:
  !pip install wandb
  import wandb

In [52]:
!openai wandb sync

'openai'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


# 1.Preparing the dataset

## 1.1. Preparing the data in JSON

In [53]:
#From Gutenberg to JSON
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
import requests
from bs4 import BeautifulSoup
import json
import re

# First, fetch the text of the book
# Option 1: from Project Gutenberg
#url = 'http://www.gutenberg.org/cache/epub/4280/pg4280.html'
#response = requests.get(url)
#soup = BeautifulSoup(response.content, 'html.parser')

# Option 2: from the GitHub repository:
#Development access to delete when going into production
!curl -H 'Authorization: token {github_token}' -L https://raw.githubusercontent.com/Denis2054/Transformers-for-NLP-and-Computer-Vision-3rd-Edition/master/Chapter08/gutenberg.org_cache_epub_4280_pg4280.html --output "gutenberg.org_cache_epub_4280_pg4280.html"

# Open and read the downloaded HTML file
with open("gutenberg.org_cache_epub_4280_pg4280.html", 'r', encoding='utf-8') as file:
    file_contents = file.read()

# Parse the file contents using BeautifulSoup
soup = BeautifulSoup(file_contents, 'html.parser')

# Get the text of the book and clean it up a bit
text = soup.get_text()
text = re.sub('\s+', ' ', text).strip()

# Split the text into sentences
sentences = sent_tokenize(text)

# Define the separator and ending
prompt_separator = " ->"
completion_ending = "\n"

# Now create the prompts and completions
data = []
for i in range(len(sentences) - 1):
    data.append({
        "prompt": sentences[i] + prompt_separator,
        "completion": " " + sentences[i + 1] + completion_ending
    })

# Write the prompts and completions to a file
with open('kant_prompts_and_completions.json', 'w') as f:
    for line in data:
        f.write(json.dumps(line) + '\n')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\an9383\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
curl: (6) Could not resolve host: token
curl: (3) URL rejected: Bad hostname
  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100  1.26M 100  1.26M   0      0  4.85M      0                              0
100  1.26M 100  1.26M   0      0  4.84M      0                              0
100  1.26M 100  1.26M   0      0  4.84M      0                              0


<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.1//EN" "http://www.w3.org/TR/xhtml11/DTD/xhtml11.dtd">
<!-- saved from url=(0053)https://www.gutenberg.org/cache/epub/4280/pg4280.html -->
<html xmlns="http://www.w3.org/1999/xhtml"><head><meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
    <title> </title>
    <meta content="text/css" http-equiv="Content-Style-Type">
    <meta content="Project Gutenberg EPUB-Maker v0.02 by Marcello Perathoner &lt;webmaster@gutenberg.org&gt;" name="generator">
    <link href="http://purl.org/dc/terms/" rel="schema.DCTERMS">
    <link href="http://www.loc.gov/loc.terms/relators/" rel="schema.MARCREL">
    <meta content="The Critique of Pure Reason" name="DCTERMS.title">
    <meta content="http://www.gutenberg.org/dirs/etext03/cprrn10.txt" name="DCTERMS.source">
    <meta content="en" scheme="DCTERMS.RFC4646" name="DCTERMS.language">
    <meta content="2010-02-15T03:18:40.059474+00:00" scheme="DCTERMS.W3CDTF" name="DCTERMS.modified">
    <meta

In [54]:
import pandas as pd

# Load the data
df = pd.read_json('kant_prompts_and_completions.json', lines=True)
df

,prompt,completion
0,The Project Gutenberg Etext of The Critique of...,Be sure to check the copyright laws for your ...
1,Be sure to check the copyright laws for your c...,"We encourage you to keep this file, exactly a..."
2,"We encourage you to keep this file, exactly as...",Please do not remove this.\n
3,Please do not remove this. ->,This header should be the first thing seen wh...
4,This header should be the first thing seen whe...,Do not change or edit it without written perm...
...,...,...
6122,"78-79. is their motto, under which they may le...",As regards those who wish to pursue a scienti...
6123,As regards those who wish to pursue a scientif...,"When I mention, in relation to the former, th..."
6124,"When I mention, in relation to the former, the...",The critical path alone is still open.\n
6125,The critical path alone is still open. ->,If my reader has been kind and patient enough...


##  1.2. Converting the data to JSONL

Answer the questions as necessary for your project.

In [55]:
!openai tools fine_tunes.prepare_data -f "kant_prompts_and_completions.json"

'openai'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [56]:
import json

# Open the file and read the lines
with open('kant_prompts_and_completions_prepared.jsonl', 'r', encoding='utf-8') as f:
    lines = f.readlines()

# Parse and print the first 5 lines
for line in lines[199:300]:
    data = json.loads(line)
    print(json.dumps(data, indent=4))


{
    "prompt": "For he found that it was not sufficient to meditate on the figure, as it lay before his eyes, or the conception of it, as it existed in his mind, and thus endeavour to get at the knowledge of its properties, but that it was necessary to produce these properties, as it were, by a positive a priori construction; and that, in order to arrive with certainty at a priori cognition, he must not attribute to the object any other properties than those which necessarily followed from that which he had himself, in accordance with his conception, placed in the object. ->",
    "completion": " A much longer period elapsed before physics entered on the highway of science.\n"
}
{
    "prompt": "A much longer period elapsed before physics entered on the highway of science. ->",
    "completion": " For it is only about a century and a half since the wise Bacon gave a new direction to physical studies, or rather\u2014as others were already on the right track\u2014imparted fresh vigour t

creating the file on openai

In [57]:
from openai import OpenAI
client = OpenAI()
print(client.files)

client.files.create(
  file=open("./kant_prompts_and_completions_prepared.jsonl", "rb"),
  purpose='fine-tune'
)



FileObject(id='file-3Yxn8jeWZrBSFvUaWapaeY', bytes=2761402, created_at=1784412543, filename='kant_prompts_and_completions_prepared.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

# 2.Fine-tuning a model

In [59]:
import os
import openai
from openai import OpenAI
client = OpenAI()

client.api_key = os.getenv("OPENAI_API_KEY")
from openai import OpenAI
client = OpenAI()

from openai import OpenAI
client = OpenAI()

client.fine_tuning.jobs.create(
  training_file='file-6R7NzCTTupQo6W5mW1C7NKKE',
  model="babbage-002"
)

PermissionDeniedError: Error code: 403 - {'error': {'message': 'OpenAI is winding down the fine-tuning platform and your organization is no longer able to create new fine-tuning training jobs. Learn more https://developers.openai.com/api/docs/deprecations#update-to-openais-self-serve-fine-tuning', 'type': 'invalid_request_error', 'param': None, 'code': 'training_not_available'}}

# 3.Running the fine-tuned GPT-3 model

We will now run the model for a completion task

Note: If your fine-tuned model does not appear immediately after the end of the fine-tuning process, you might have to wait until it is processed by OpenAI.

1.Go to the OpenAI Playground to test your model: https://platform.openai.com/playground

2.Then implement it in your environment

In [60]:
# List all created fine-tunes
from openai import OpenAI
client = OpenAI()
client.fine_tuning.jobs.list


<bound method Jobs.list of <openai.resources.fine_tuning.jobs.jobs.Jobs object at 0x000002102C1109D0>>

[Consult OpenAI fine-tune documentation for more](https://platform.openai.com/docs/guides/fine-tuning/create-a-fine-tuned-model)